In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.animation as animation
import seaborn as sb
from numpy import linalg as LA
from scipy.integrate import odeint
from multiprocessing import Pool
import xgi
import networkx as nx
mpl.use('Qt5Agg') 
plt.ion()

plt.rcParams['font.sans-serif'] = ['Times New Roman', 'SimHei']
plt.rcParams['mathtext.fontset'] = 'cm'        
plt.rcParams['axes.unicode_minus'] = False


In [ ]:
def kuramoto(N, t_end, dt, r1, r2, k1, k2, initial, alpha, beta):
    """Simulates Kuramoto oscillators on a ring with higher-order interactions (modulus version)."""
    def rhs(theta, N, r1, r2, k1, k2, alpha, beta):
        dtheta_dt = np.zeros(N)
        local_order = np.zeros(N, dtype=np.complex64)
        phases = theta

        # Compute local order parameter
        for ii in range(N):
            for j in range(-r2, r2 + 1):
                idx = (ii + j) % N
                local_order[ii] += np.exp(1j * phases[idx])

        local_order /= (2 * r2 + 1)
        epsilon = 1e-10
        threshold = 1e-12
        local_order[np.abs(local_order) < threshold] = epsilon

        # Use the modulus of the local order parameter (real-valued adaptive factor)
        O_alpha = np.abs(local_order) ** alpha
        O_beta = np.abs(local_order) ** beta

        # Pairwise coupling
        for ii in range(N):
            for jj in range(-r1, r1 + 1):
                dtheta_dt[ii] += O_alpha[ii] * k1 * (np.sin(theta[(ii + jj) % N] - theta[ii])) / r1

        # Triadic coupling
        idx = list(range(-r2, 0)) + list(range(1, r2 + 1))
        for ii in range(N):
            for jj in idx:
                for kk in idx:
                    if jj != kk:
                        dtheta_dt[ii] += (
                            k2 * O_beta[ii]
                            * np.sin(theta[(ii + kk) % N] + theta[(ii + jj) % N] - 2 * theta[ii])
                            / (r2 * (2.0 * r2 - 1))
                        )
        return dtheta_dt, local_order, O_alpha, O_beta

    times = np.arange(0, t_end + dt / 2, dt)
    n_t = len(times)
    thetas = np.zeros((N, n_t))
    O_alphas = np.zeros((N, n_t)) 
    O_betas = np.zeros((N, n_t))
    O_s = np.zeros((N, n_t))
    thetas[:, 0] = initial
    for it in range(1, n_t):
        dtheta_dt, local_order, O_alpha, O_beta = rhs(thetas[:, it - 1], N, r1, r2, k1, k2, alpha, beta)
        thetas[:, it] = thetas[:, it - 1] + dt * dtheta_dt
        O_alphas[:, it] = O_alpha
        O_betas[:, it] = O_beta
        O_s[:, it] = np.abs(local_order)  

    return thetas, O_s, O_alphas, O_betas, times


In [ ]:
def identify_winding_number(xa, N, t):
    q, Delta = 0, np.zeros(N)
    for ii in range(N):
        delta = xa[(ii + 1) % N, t] - xa[ii, t]
        if delta > np.pi:
            delta -= 2 * np.pi
        elif delta < -np.pi:
            delta += 2 * np.pi
        q += delta
        Delta[ii] = delta

    w_no = round(q / (2 * np.pi))
    Is_twisted_state = LA.norm(Delta - np.mean(Delta)) < 1e-1
    return w_no, Is_twisted_state


In [ ]:

def detect_twist_transition(thetas, N, times, atol=1e-2):
    def is_twisted_q(t_idx):
        q, is_q = identify_winding_number(thetas, N, t_idx)
        return q, is_q

    initial_q, is_twisted_start = is_twisted_q(0)
    final_q, is_twisted_end = is_twisted_q(-1)

    if is_twisted_start and is_twisted_end and initial_q == final_q:
        return 1, initial_q, 0.0, 0.0 

    elif is_twisted_end:
        left_time, right_time = None, None

        for t_idx in range(1, thetas.shape[1]):
            q_t, is_q = is_twisted_q(t_idx)
            if not (is_q and q_t == initial_q):   
                left_time = times[t_idx]
                break

        for t_idx in range(1, thetas.shape[1]):
            q_t, is_q = is_twisted_q(t_idx)
            if is_q and q_t == final_q:
                right_time = times[t_idx]
                break

        if left_time is not None and right_time is not None:
            transition_duration = right_time - left_time
            return 0, final_q, left_time, transition_duration
        else:
            return 0, final_q, left_time, None  

    else:
        return 2, None, left_time, None 


In [ ]:
def plot_phases_ring(H, thetas, it, ax=None, colorbar=False, 
                     cbar_fraction=0.03, cbar_pad=0.02, cbar_shrink=1.0, cbar_aspect=10, **kwargs):
    if ax is None:
        ax = plt.gca()

    pos = xgi.circular_layout(H) 
    psi = thetas[:, it] % (2 * np.pi) 

    ax, im = xgi.draw_nodes(
        H,
        pos=pos,
        ax=ax,
        node_fc=psi,
        vmin=0,
        vmax=2 * np.pi,
        node_fc_cmap="coolwarm",
        **kwargs,
    )

    ax.set_aspect("equal") 
    return ax, im


In [ ]:
def compute_sigma_Delta(thetas, N, R):
    """Computes the incoherence strength at the final time step, considering phase periodicity."""
    epsilon=1e-4
    theta_final = thetas[:, -1]  
    sigma_Delta = np.zeros(N)

    delta_x = np.roll(theta_final, -1) - theta_final  
    delta_x = (delta_x + np.pi) % (2 * np.pi) - np.pi  

    sigma_Delta = np.zeros(N)
    for x in range(N):
        neighbors = [(x + j) % N for j in range(-R, R)] 
        delta_x_mean = np.mean(delta_x[neighbors])
        sigma_Delta[x] = np.sqrt((3 / (2 * R)) * np.sum((delta_x[neighbors] - delta_x_mean) ** 2))

    sigma_Delta /= np.max(sigma_Delta)
    sigma_Delta_final = np.mean(sigma_Delta)
    if sigma_Delta_final < epsilon:
        sigma_Delta_final = 0.0

    return np.mean(sigma_Delta_final)  

In [ ]:

N = 83
r1, r2 = 2, 2
k1 = 1
t_end = 200
dt = 0.01
alpha = 0
param_sets = [(13, 1, 0), (13, 1, 1)]

for idx, (Q, k2, beta) in enumerate(param_sets):
    H = xgi.trivial_hypergraph(N)
    sync = np.linspace(0, 2 * np.pi * Q, N + 1)[:-1]
    sync += np.pi if Q == 0 else 0
    p = 2 * np.pi * (np.random.rand(N) - 0.5)
    p -= np.mean(p)
    sync = sync + 1e-5 * p

    thetas, O_s, O_alphas, O_betas, times = kuramoto(N, t_end, dt, r1, r2, k1, k2, sync, alpha, beta)
    thetas = thetas % (2 * np.pi)

    np.save(f"thetas_Q{Q}_k2_{k2}_beta_{beta}.npy", thetas)
    np.save(f"times_Q{Q}_k2_{k2}_beta_{beta}.npy", times)

In [ ]:
plt.close('all')
zz = 16
param_sets = [(13, 1, 0), (13, 1, 1)]
labels = ["(a)", "(b)", "(c)", "(d)"]
N = 83

fig, axes = plt.subplots(2, 2, figsize=(5, 3.5), gridspec_kw={'wspace': 0.2, 'hspace': 0.3})

contour_for_cbar = None

for idx, (Q, k2, beta) in enumerate(param_sets):
    H = xgi.trivial_hypergraph(N)
    thetas = np.load(f"thetas_Q{Q}_k2_{k2}_beta_{beta}.npy")
    winding_number, is_twisted = identify_winding_number(thetas, N, -1)
    print(f"Q = {Q}, k2 = {k2}, beta = {beta}")
    print(f"Last time step winding number: {winding_number}")
    times = np.load(f"times_Q{Q}_k2_{k2}_beta_{beta}.npy")
    

    downsample_factor = 100
    thetas_downsampled = thetas[:, ::downsample_factor]
    times_downsampled = times[::downsample_factor]
    time_grid, node_grid = np.meshgrid(times_downsampled, np.arange(N))

    ax1 = axes[idx, 0]
    contour = ax1.contourf(time_grid, node_grid, thetas_downsampled, levels=100,
                           cmap="coolwarm", vmin=0, vmax=2 * np.pi)
    if idx == 0:
        contour_for_cbar = contour  

    ax1.set_xlabel(r"$\it{Time}$", fontsize=zz)
    ax1.set_ylabel(r"$\it{i}$", fontsize=zz)
    ax1.set_yticks([0, 40, 80])
    ax1.tick_params(axis='both', labelsize=zz-2)

    ax2 = axes[idx, 1]
    plot_phases_ring(H, thetas, -1, ax=ax2, node_size=5, alpha=0.8, node_lw=0.2,
                     colorbar=False) 

    ax2.tick_params(axis='both', labelsize=zz - 2)

    ax1.text(0.1, 1.05, labels[2 * idx], transform=ax1.transAxes, fontsize=zz,
             va="bottom", ha="right")
    ax2.text(0.1, 1.05, labels[2 * idx + 1], transform=ax2.transAxes, fontsize=zz,
             va="bottom", ha="right")

    if idx == 0:
        ax1.set_xticks([])
        ax1.set_xlabel("")

cax = fig.add_axes([axes[0, 1].get_position().x1 + 0.02,
                    axes[1, 1].get_position().y0,
                    0.012,
                    axes[0, 1].get_position().y1 - axes[1, 1].get_position().y0])

cbar = plt.colorbar(contour_for_cbar, cax=cax, ticks=[0, np.pi, 2 * np.pi])
cbar.ax.set_title(r"$\theta_i$", fontsize=zz, fontweight="bold", pad=5)
cbar.set_ticklabels([r"$0$", r"$\pi$", r"$2\pi$"])
cbar.ax.tick_params(labelsize=zz-1)

plt.tight_layout()
# plt.savefig("fig1.eps", format="eps",dpi=300, bbox_inches="tight", pad_inches=0)
plt.show()